# IOAI — 2025 Stage 1 Hidden Subsequences (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/OlimpiadaAI/II-OlimpiadaAI"
CLONE = "II-OlimpiadaAI"
SUBDIR = "1_etap/5_ukryte_podciagi"
WORKDIR = "1_etap/5_ukryte_podciagi"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# Ukryte Podciągi
![](https://live.staticflickr.com/65535/54327633282_6bc45ba42a_o.png)

*Obraz wygenerowany przy użyciu modelu DALL-E.*

## Wstęp
Od starożytnych wróżbitów interpretujących układ gwiazd po współczesnych kryptografów tropiących ślady ukrytych wiadomości, ludzkość od zawsze poszukiwała sensu w pozornie chaotycznych danych. Czasem kluczowe informacje ukrywają się w drobnych sekwencjach symboli, a ich wartość ujawnia się dopiero po precyzyjnej analizie.

W tym zadaniu wcielisz się w rolę detektywa, poszukującego strukturalnych zależności w zbiorze binarnych ciągów. Będziesz dysponował zbiorem danych zawierającym przykładowe ciągi oraz ich poprawnie obliczone wartości. Twoim celem będzie opracowanie metody analizy ukrytych wzorców pozwalającej na możliwie najprecyzyjniejsze wyznaczanie wartości ciągów niewystępujących w zbiorze.

Mówimy, że ciąg $T$ jest podciągiem $S$ i oznaczamy $T \subseteq S$ jeżeli
$$
T = S_{i_1}S_{i_2} \dots S_{i_k}
$$
Gdzie
$$
1 \leq i_1 < i_2 < \dots < i_k \leq n
$$
Dla $k$ i $n$ będących długościami ciągów $T$ i $S$ odpowiednio, oraz indeksów ($i_1 < i_2 < \dots < i_k$) będących ściśle rosnącym ciągiem liczb naturalnych (niekoniecznie kolejnych).

Rozwiązaniem dla danego ciągu binarnego $S \in \{0,1\}^{n}$ oraz zdefiniowanego zbioru zawierającego kolejno wzorzec i jego wagę $W = \{(T, v):T \in \{0,1\}^{k}, k \le n, v \in Z\}$ jest liczba
$$
\phi(S) = \sum_{(T_{i}, v_{i}) \in W} v_{i} \cdot \text{I} (T_{i})
$$
gdzie $\text{I}(T_{i}) = \begin{cases}
1, & T_{i} \subseteq S \\
0, & T_{i} \subsetneq S
\end{cases}$

Innymi słowy, $\phi(S)$ jest sumą wartości wszystkich ciągów ze zbioru $W$, które są podciągami $S$.

**Przykład:**
Dla zbioru $W = \{(1111, 1), (1010, 2)\}$, mamy:
- $\phi$(0**1111**000) = 1
- $\phi$(1**10**00**1**0**0**) = 2
- $\phi$(0**110110**0) = 3,  ponieważ

  - 1111 $\subseteq$ 0**11**0**11**00

  - 1010 $\subseteq$ 01**101**1**0**0
- $\phi$(01100000) = 0, ponieważ 1111, 1010 $\subsetneq$ 01100000.


## Zadanie
Stwórz model (obiekt typu `nn.Module`) który będzie znajdował wartość $\phi$ dla ciągów ze zbioru danych. Dane treningowe składają się z ciągów $S$ oraz odpowiadających im wartości $\phi(S)$. Zauważ więc, że wzorzec $W$ jest ukryty, a Twoim zadaniem jest przybliżyć $\phi$ bez wiedzy o nim.

Twój model musi przyjmować na wejściu dane w postaci $(\text{batch}, n)$. Natomiast na wyjściu musi zwracać wartości w postaci $(\text{batch}, 1)$ lub $(\text{batch},)$, gdzie $\text{batch}$ to liczba próbek.

### Dane
Dane dostępne dla Ciebie w tym zadaniu to:
* `train_dataset.csv`- plik z danymi na których będziesz trenować swój model
* `val_dataset.csv`- plik z danymi na których przetestujesz swój model

### Kryterium Oceny
Zadanie zostanie ocenione na podstawie metryki [MSE](https://en.wikipedia.org/wiki/Mean_squared_error) (Mean Squared Error), która jest jedną z najczęściej stosowanych metryk do oceny jakości regresji.  

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
Gdzie $y_i$ to wartość rzeczywista, a $\hat{y}_i$ to wartość przewidziana przez model. Wartość $i$ to numer próbki, a $n$ to liczba wszystkich próbek.

Ta metryka jest już przez nas zaimplementowana w tym notebooku.

**Ostatecznie Twoje rozwiązanie oceniane będzie na tajnym zbiorze testowym na podstawie metryki MSE.** Zbiór testowy nie różni się znacząco od zbioru walidacyjnego.

- Gdy wartość MSE dla Twojego modelu będzie wynosiła 64 (lub więcej), otrzymasz 0 punktów za zadanie
- Gdy wartość MSE dla Twojego modelu będzie wynosiła 64 (lub mniej), otrzymasz X punktów za zadanie, gdzie X jest zdefiniowane w następujący sposób:
$$
\text{X} = \frac{64 - MSE}{64} \times 100
$$

## Ograniczenia
- Twoje rozwiązanie powinno zawierać model ML/DL z wyuczalnymi parametrami. Rozwiązania czysto algorytmiczne nie będą akceptowane.
- Twoje rozwiazanie będzie testowane na Platformie Konkursowej bez dostępu do internetu oraz w środowisku z GPU.
- Ewaluacja Twojego finalnego rozwiązania na Platformie Konkursowej nie może trwać dłużej niż 4 minut z GPU.
- Twój model może być trenowany maksymalnie przez 4000 iteracji, co odpowiada pojedynczemu przeiterowaniu przez zmienną ```dl``` (patrz na przykładowe rozwiązanie).
- Twój model nie może mieć więcej niż 50000 parametrów.

## Uwagi i Wskazówki
- Każdy z ciągów ma taką samą, ustaloną długość.
- Każdy z szukanych podciągów ma długość krótszą niż długość źródłowych ciągów.
- Rozpatrujemy trzy podciągi. Każdy z nich ma przypisaną wartość, która jest liczbą całkowitą.
- W każdym ciągu znajduje się dowolna liczba podciągów (w tym brak jakiegokolwiek z nich).
- Ciągi i podciągi pochodzą z alfabetu binarnego oraz są reprezentowane w postaci list.

## Pliki Zgłoszeniowe
Ten notebook uzupełniony o Twoje rozwiązanie (patrz klasa `YourModel` oraz trening modelu).

## Ewaluacja
Pamiętaj, że podczas sprawdzania flaga `FINAL_EVALUATION_MODE` zostanie ustawiona na `True`.

Za to zadanie możesz zdobyć pomiędzy 0 a 100 punktów. Liczba punktów, którą zdobędziesz, będzie wyliczona na (tajnym) zbiorze testowym na Platformie Konkursowej na podstawie wyżej wspomnianego wzoru, zaokrąglona do liczby całkowitej. Jeśli Twoje rozwiązanie nie będzie spełniało powyższych kryteriów lub nie będzie wykonywać się prawidłowo, otrzymasz za zadanie 0 punktów.

# Kod Startowy

W tej sekcji inicjalizujemy środowisko poprzez zaimportowanie potrzebnych bibliotek i funkcji. Przygotowany kod ułatwi Tobie efektywne operowanie na danych i budowanie właściwego rozwiązania.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

FINAL_EVALUATION_MODE = False # Podczas sprawdzania ustawimy tą flagę na True.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

import os
import gdown
import pandas
import torch
import numpy as np
import torch.optim as optim
import torch.nn as nn

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

def seed_everything(seed: int) -> None:
    """
    Ustawia ziarno (seed) dla odtwarzalności wyników w Pythonie, NumPy oraz PyTorch.

    Funkcja ustawia ziarno dla generatorów liczb losowych Pythonie, NumPy oraz PyTorch,
    a także konfiguruje PyTorch do pracy w trybie deterministycznym.

    Parametry:
        seed (int): Wartość ziarna do ustawienia.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

seed_everything(12345)

device = 'cuda'
assert torch.cuda.is_available(), "CUDA niedostępna!"

## Ładowanie Danych
Za pomocą poniższego kodu wczytujemy dane zawierające ciągi wraz z ich wartościami.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

class CSVDataloader(torch.utils.data.DataLoader):
    """
    Klasa CSVDataloader służy do ładowania danych z plików CSV i zwracania ich w batchach.

    Przyjmuje: 
        csv_file (str): Ścieżka do pliku CSV.
        batch_size (int): Rozmiar batcha.
        shuffle (bool): Czy tasować dane.
    """
    def __init__(self, csv_file, batch_size=128, shuffle=True):
        
        class CSVDataset(torch.utils.data.Dataset):
            """
            Klasa CSVDataset służy do przechowywania danych z plików CSV jako pojedyncze próbki. 
            """
            def __init__(self, csv_file: str):
                data = pandas.read_csv(csv_file).values
                self.x = torch.tensor(data[:, :-1], dtype=torch.float32)  # Features
                self.y = torch.tensor(data[:, -1], dtype=torch.float32)  # Labels

            def __len__(self) -> int:
                return len(self.x)

            def __getitem__(self, idx: int) -> tuple:
                return self.x[idx].long(), self.y[idx]

        dataset = CSVDataset(csv_file)
        self.seq_len = dataset.x.shape[1]
        super().__init__(dataset, batch_size=batch_size, shuffle=shuffle)

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# Inicjalizacja treningowego zbioru danych
train_dataset_path = "train_dataset.csv"
val_dataset_path = "val_dataset.csv"

if not os.path.exists(train_dataset_path):
    url = "https://drive.google.com/uc?id=1INeYNpPA_YwojuQbMizlsFsERJ-PJX-E"
    gdown.download(url, train_dataset_path, quiet=True)

if not os.path.exists(val_dataset_path):
    url = "https://drive.google.com/uc?id=1oQcOMyDWVX0x76LOyp4TcFip1koRuodN"
    gdown.download(url, val_dataset_path, quiet=True)

dl = CSVDataloader("train_dataset.csv")
val_dl = CSVDataloader("val_dataset.csv")

## Kod z Kryterium Oceniającym

Kod, zbliżony do poniższego, będzie używany do oceny rozwiązania na zbiorze testowym.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

def mse_criterium(
        estimations: torch.Tensor, 
        answers: torch.Tensor
    ) -> torch.Tensor:
    """
    Oblicza wartość funkcji błędu średniokwadratowego (MSE) pomiędzy predykcjami a prawdziwymi wartościami.

    Parametry:
        estimations (torch.Tensor): Predykcje modelu.
        answers (torch.Tensor): Prawdziwe wartości.

    Zwraca:
        torch.Tensor: Wartość funkcji błędu średniokwadratowego.
    """
    return torch.mean((estimations.view(-1) - answers.view(-1)) ** 2)


def validate_model(
        model: torch.nn.Module, 
        val_dl: torch.utils.data.DataLoader,
    ) -> float:
    """
    Waliduje model na zbiorze walidacyjnym. Zwraca uśrednioną wartość 
    funkcji błędu średniokwadratowego dla wszystkich próbek.

    Parametry:
        model (torch.nn.Module): Model do oceny.
        val_dl (torch.utils.data.DataLoader): DataLoader z danymi walidacyjnymi.

    Zwraca:
        float: Uśredniona wartość funkcji błędu średniokwadr
    """
    model = model.eval().to(device)
    values = []
    for x, y in val_dl:
        x = x.to(device)
        y = y.to(device)
        y_pred = model(x)

        mse = mse_criterium(y_pred, y).cpu().item()
        values.append(mse)

    final_value = torch.tensor(values).mean().item()

    return final_value

def estimate_points(mse: float) -> int:
    """
    Funkcja wyznaczająca ilość punktów za zadanie na podstawie wartości funkcji błędu średniokwadratowego.

    Parametry:
        mse (float): Wartość funkcji błędu średniokwadratowego.

    Zwraca:
        int: Ilość punktów za zadanie (0 - 100).
    """
    points = max((100 * (64 - mse)) / 64, 0)
    return int(round(points))

## Przykładowe Rozwiązanie
Poniżej przedstawiamy uproszczone rozwiązanie, które służy jako przykład demonstrujący podstawową funkcjonalność notatnika. Może ono posłużyć jako punkt wyjścia do opracowania Twojego rozwiązania.

Jako prosty przykład może posłuży rozwiązanie bazujące na wielowarstwowej sieci neuronowej (ang. Multi-layered perceptron, MLP).
W tym wypadku, ciągi zer i jedynek traktujemy jako wejście do naszej sieci, natomiast jej wyjściem modelujemy wartość danego ciągu. Minimalizując błąd średniokwadratowy (ang. mean squared error, MSE) uczymy sieć prawidłowo oszacowywać wartość podciągu na bazie jego elementów. 

Poniższa ilustracja ukazuje w jaki sposób uczymy nasz model prawidłowo oceniać wartości ciągów.

![](https://live.staticflickr.com/65535/54328760659_2e9355bb07_c.jpg)

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

class MLP(nn.Module):
    """
    Klasa reprezentująca model sieci MLP z czterema ukrytymi warstwami.

    Parametry:
        input_length (int): Długość wejścia sieci (długość sekwencji). 
    """
    def __init__(self, input_length: int):
        super(MLP, self).__init__()
        neurons_num = [256, 128, 64, 32]

        self.fc_layers = nn.Sequential(
            nn.Linear(input_length, neurons_num[0]),
            nn.ReLU(),
            nn.Linear(neurons_num[0], neurons_num[1]),
            nn.ReLU(),
            nn.Linear(neurons_num[1], neurons_num[2]),
            nn.ReLU(),
            nn.Linear(neurons_num[2], neurons_num[3]),
            nn.ReLU(),
            nn.Linear(neurons_num[3], 1),
        )

        print("Liczba parametrów:", sum(p.numel() for p in self.parameters()))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Funkcja przyjmująca sekwencje danych i zwracająca predykcje ich wartości za pomocą sieci MLP.

        Parametry:
            x (torch.Tensor): Sekwencja danych.

        Zwraca:
            torch.Tensor: Predykcje wartości sekwencji.
        """
        x = x.float()
        x = self.fc_layers(x)
        return x


### Trening Przykładowego Modelu

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

if not FINAL_EVALUATION_MODE:
	model = MLP(dl.seq_len).to(device)

	optimizer = optim.Adam(model.parameters(), lr=0.005)
	criterion = nn.MSELoss()

	model.train()
	for batch in iter(dl): # pojedyncze przeiterowanie dl - 4000 batchy
		inputs, targets = batch
		inputs, targets = inputs.to(device).long(), targets.to(device).float().unsqueeze(1)

		optimizer.zero_grad()
		outputs = model(inputs)

		loss = criterion(outputs, targets)
		loss.backward()
		optimizer.step()


### Ewaluacja Przykładowego Rozwiązania

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# walidacja przykładowego rozwiązania
if not FINAL_EVALUATION_MODE:
    score = validate_model(model, val_dl)
    print(f"Błąd średniokwadratowy: {score:.2f}")

# Twoje Rozwiązanie
W tej sekcji należy umieścić Twoje rozwiązanie. Wprowadzaj zmiany wyłącznie tutaj!

In [ ]:
# przykładowy model, który możesz modyfikować 

class YourModel(nn.Module):
    def __init__(self, sequence_len):
        super(YourModel, self).__init__()
        self.layer = nn.Linear(sequence_len, 1)

    def forward(self, x):
        """
        Funkcja przyjmująca sekwencje danych i zwracająca predykcje ich wartości.

        Parametry:
            x (torch.Tensor): Sekwencja danych. 

        Zwraca:
            torch.Tensor: Predykcje wartości sekwencji.
        """
        return self.layer(x.float())


### Trening Twojego Modelu
Tutaj zaimplementuj trening Twojego modelu.

In [ ]:
your_model = YourModel(dl.seq_len).to(device)

# ...

your_model = your_model.eval()

# Ewaluacja

Uruchomienie poniższej komórki pozwoli sprawdzić, ile punktów zdobyłoby Twoje rozwiązanie na danych walidacyjnych. Przed wysłaniem upewnij się, że cały notebook wykonuje się od początku do końca bez błędów i bez konieczności ingerencji użytkownika po wybraniu opcji "Run All".

In [ ]:
# ######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

if not FINAL_EVALUATION_MODE:
    assert sum(p.numel() for p in your_model.parameters()) < 50000, "Model posiada za dużo parametrów"

    mse = validate_model(your_model, val_dl)
    score = estimate_points(mse)

    print(f"Błąd średniokwadratowy: {mse:.2f}")
    print(f"Estymowane punkty za zadanie: {score}")

Podczas sprawdzania model zostanie zapisany jako `your_model.pkl` i oceniony na zbiorze testowym.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

if FINAL_EVALUATION_MODE:
    import cloudpickle

    OUTPUT_PATH = "file_output"
    FUNCTION_FILENAME = "your_model.pkl"
    FUNCTION_OUTPUT_PATH = os.path.join(OUTPUT_PATH, FUNCTION_FILENAME)

    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    your_model = your_model.eval()

    with open(FUNCTION_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(your_model, f)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)